## 1. Check that the TOML parser works

I start with valid, nested, and invalid TOML examples. This checks the parser before fuzzing.

In [1]:

import tomllib

test_cases = [
    ("Valid TOML", 'name = "Alice"\nage = 25'),
    ("Nested TOML", 'student = { name = "Bob", scores = [80, 90, 95] }'),
    ("Nested tables", '[student]\nname = "Bob"\n[student.details]\nage = 25'),
    ("Invalid TOML", 'name = "Alice"\nage = ')
]

for name, test_input in test_cases:
    print("\n" + "=" * 50)
    print(name)
    print("Input:", test_input)

    try:
        result = tomllib.loads(test_input)
        print("Status: ACCEPTED")
        print("Parsed output:", result)

    except tomllib.TOMLDecodeError as error:
        print("Status: REJECTED")
        print("Error:", type(error).__name__)
        print("Details:", error)



Valid TOML
Input: name = "Alice"
age = 25
Status: ACCEPTED
Parsed output: {'name': 'Alice', 'age': 25}

Nested TOML
Input: student = { name = "Bob", scores = [80, 90, 95] }
Status: ACCEPTED
Parsed output: {'student': {'name': 'Bob', 'scores': [80, 90, 95]}}

Nested tables
Input: [student]
name = "Bob"
[student.details]
age = 25
Status: ACCEPTED
Parsed output: {'student': {'name': 'Bob', 'details': {'age': 25}}}

Invalid TOML
Input: name = "Alice"
age = 
Status: REJECTED
Error: TOMLDecodeError
Details: Invalid value (at end of document)


## 2. Define the grammar we will use

This is a simplified TOML grammar for the experiment. It includes key-value pairs, arrays, inline tables, and nested tables. The same grammar will be used by the normal and heuristic fuzzers.

In [2]:

# Simplified TOML grammar for our project

TOML_GRAMMAR = {

    "<start>": ["<document>"],

    "<document>": [
        "title = <string>",
        "count = <integer>",
        "enabled = <boolean>",
        "items = <array>",
        "settings = <inline_table>",
        "title = <string>\ncount = <integer>",
        "title = <string>\nenabled = <boolean>",
        "title = <string>\nitems = <array>",
        "title = <string>\nsettings = <inline_table>",
        "title = <string>\nitems = <array>\nsettings = <inline_table>",
        "[owner]\nname = <string>\nage = <integer>",
        "[database]\nserver = <string>\nport = <integer>\n[database.connection]\nmax_connections = <integer>\nenabled = <boolean>"
    ],

    "<value>": [
        "<string>",
        "<integer>",
        "<float>",
        "<boolean>",
        "<array>",
        "<inline_table>"
    ],

    "<array>": [
        "[]",
        "[<elements>]",
        "[<elements>,]"
    ],

    "<elements>": [
        "<value>",
        "<value>, <elements>"
    ],

    "<inline_table>": [
        "{}",
        "{ name = <value> }",
        "{ name = <value>, active = <boolean> }",
        "{ name = <value>, details = <inline_table> }",
        "{ values = <array> }",
        "{ name = <value>, values = <array> }"
    ],

    "<string>": [
        '"Alice"',
        '"Bob"',
        '"Hello TOML"'
    ],

    "<integer>": [
        "0", "1", "7", "25", "42", "-3"
    ],

    "<float>": [
        "0.5", "3.14", "-2.5"
    ],

    "<boolean>": [
        "true", "false"
    ]
}

print("TOML grammar created successfully!")
print("Number of grammar rules:", len(TOML_GRAMMAR))


TOML grammar created successfully!
Number of grammar rules: 10


## 3. Set up the normal grammar fuzzer

I use `GrammarFuzzer` from *The Fuzzing Book*. The random seed makes the run reproducible, and `max_nonterminals=10` controls expansion size; it is not a strict depth limit.

In [3]:

from fuzzingbook.GrammarFuzzer import GrammarFuzzer
import random

random.seed(42)

fuzzer = GrammarFuzzer(
    TOML_GRAMMAR,
    max_nonterminals=10
)

print("Grammar fuzzer initialized successfully!")


Grammar fuzzer initialized successfully!


## 4. Generate the first batch of normal inputs

The normal fuzzer generates 2,000 inputs and stores them in `generated_inputs`. I print the inputs to inspect what the grammar produces.

In [4]:

generated_inputs = []

for i in range(2000):
    test_input = fuzzer.fuzz()
    generated_inputs.append(test_input)

    print(f"Input {i + 1}: {test_input}")


Input 1: [database]
server = "Alice"
port = 1
[database.connection]
max_connections = 0
enabled = true
Input 2: title = "Alice"
settings = { name = 1 }
Input 3: title = "Bob"
count = 0
Input 4: [owner]
name = "Hello TOML"
age = -3
Input 5: count = 25
Input 6: [owner]
name = "Bob"
age = -3
Input 7: title = "Bob"
items = []
settings = { name = { name = 42 }, values = [[true], -2.5] }
Input 8: count = 0
Input 9: count = -3
Input 10: title = "Hello TOML"
settings = { name = "Bob" }
Input 11: title = "Alice"
enabled = false
Input 12: enabled = true
Input 13: title = "Hello TOML"
settings = { name = false }
Input 14: settings = { name = 0.5, details = {} }
Input 15: count = 25
Input 16: title = "Hello TOML"
items = ["Alice", 3.14, 0.5]
settings = {}
Input 17: title = "Bob"
count = 1
Input 18: items = [{ name = false, values = [] }]
Input 19: enabled = true
Input 20: [database]
server = "Hello TOML"
port = 42
[database.connection]
max_connections = 1
enabled = true
Input 21: title = "Hello TO

## 5. Test the first batch against TOML

I send the generated inputs to `tomllib.loads()` and record whether each one is accepted or rejected. Unexpected exceptions are counted separately.

In [5]:

results = []

for i, test_input in enumerate(generated_inputs, start=1):

    try:
        parsed = tomllib.loads(test_input)
        status = "ACCEPTED"
        error = None

    except tomllib.TOMLDecodeError as e:
        status = "REJECTED"
        error = str(e)

    except Exception as e:
        status = "UNEXPECTED ERROR"
        error = f"{type(e).__name__}: {e}"

    results.append({
        "input": test_input,
        "status": status,
        "error": error
    })

    print(f"Test {i}: {status}")
    print(f"Input: {test_input}")

    if error is not None:
        print(f"Error: {error}")

    print("-" * 40)

total = len(results)
accepted = sum(r["status"] == "ACCEPTED" for r in results)
rejected = sum(r["status"] == "REJECTED" for r in results)
unexpected = sum(r["status"] == "UNEXPECTED ERROR" for r in results)

print("\n--- TEST SUMMARY ---")
print("Total inputs:", total)
print("Accepted:", accepted)
print("Rejected:", rejected)
print("Unexpected errors:", unexpected)
print("Acceptance rate:", f"{accepted / total * 100:.2f}%")


Test 1: ACCEPTED
Input: [database]
server = "Alice"
port = 1
[database.connection]
max_connections = 0
enabled = true
----------------------------------------
Test 2: ACCEPTED
Input: title = "Alice"
settings = { name = 1 }
----------------------------------------
Test 3: ACCEPTED
Input: title = "Bob"
count = 0
----------------------------------------
Test 4: ACCEPTED
Input: [owner]
name = "Hello TOML"
age = -3
----------------------------------------
Test 5: ACCEPTED
Input: count = 25
----------------------------------------
Test 6: ACCEPTED
Input: [owner]
name = "Bob"
age = -3
----------------------------------------
Test 7: ACCEPTED
Input: title = "Bob"
items = []
settings = { name = { name = 42 }, values = [[true], -2.5] }
----------------------------------------
Test 8: ACCEPTED
Input: count = 0
----------------------------------------
Test 9: ACCEPTED
Input: count = -3
----------------------------------------
Test 10: ACCEPTED
Input: title = "Hello TOML"
settings = { name = "Bob" 

## 6. Measure normal grammar production coverage

I use `TrackingGrammarCoverageFuzzer` to generate 1,000 inputs and record which grammar productions were used. This measures grammar coverage, not parser code coverage.

In [6]:

from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer
import random

random.seed(42)
N = 1000

normal_fuzzer = TrackingGrammarCoverageFuzzer(
    TOML_GRAMMAR,
    max_nonterminals=10
)

normal_inputs = [normal_fuzzer.fuzz() for _ in range(N)]

used = normal_fuzzer.expansion_coverage()
total = normal_fuzzer.max_expansion_coverage()

normal_grammar_coverage = len(used) / len(total) * 100

print("Generated inputs:", len(normal_inputs))
print("Total grammar productions:", len(total))
print("Used productions:", len(used))
print(f"Grammar coverage: {normal_grammar_coverage:.2f}%")
print("Missing productions:", normal_fuzzer.missing_expansion_coverage())


Generated inputs: 1000
Total grammar productions: 44
Used productions: 44
Grammar coverage: 100.00%
Missing productions: set()


## 7. Measure normal parser code coverage

I use `coverage.py` while the TOML parser processes the 1,000 normal inputs. `cover_pylib=True` allows coverage.py to include the Python standard-library parser.

In [7]:

import coverage
import tomllib

cov = coverage.Coverage(
    source=["tomllib"],
    branch=False,
    data_file=None,
    cover_pylib=True
)

cov.start()

try:
    for test_input in normal_inputs:
        try:
            tomllib.loads(test_input)
        except tomllib.TOMLDecodeError:
            pass
finally:
    cov.stop()

print("NORMAL FUZZER - CODE COVERAGE")
normal_code_coverage = cov.report()

print(f"\nTotal line coverage: {normal_code_coverage:.2f}%")


NORMAL FUZZER - CODE COVERAGE
Name                                                                            Stmts   Miss  Cover
---------------------------------------------------------------------------------------------------
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\__init__.py       3      3     0%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_parser.py      464    254    45%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_re.py           35     32     9%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_types.py         4      4     0%
---------------------------------------------------------------------------------------------------
TOTAL                                                                             506    293    42%

Total line coverage: 42.09%


f:\Study ITU\Practical Software Analysis\project\.venv\Lib\site-packages\coverage\inorout.py:576: CoverageWarning: Module tomllib was previously imported, but not measured (module-not-measured); see https://coverage.readthedocs.io/en/7.17.3/messages.html#warning-module-not-measured
  self.warn(msg, slug="module-not-measured")


## 8. Create the heuristic-guided grammar fuzzer

This class changes how grammar alternatives are selected. Arrays and inline tables receive higher weights, encouraging more structured and nested TOML values.

In [8]:

import random
from fuzzingbook.GrammarFuzzer import GrammarFuzzer

class HeuristicGrammarFuzzer(GrammarFuzzer):

    def choose_node_expansion(self, node, children_alternatives):
        symbol, tree = node
        weights = []

        for children in children_alternatives:
            expansion = "".join(child[0] for child in children)

            if symbol == "<value>" and expansion in (
                "<inline_table>",
                "<array>"
            ):
                weights.append(4)
            else:
                weights.append(1)

        return random.choices(
            range(len(children_alternatives)),
            weights=weights
        )[0]

print("Heuristic fuzzer created successfully!")


Heuristic fuzzer created successfully!


## 9. Generate heuristic-guided inputs

I use the same grammar, random seed, input count, and expansion setting as the normal fuzzer. Only the production-selection strategy changes.

In [9]:

random.seed(42)
N = 1000

heuristic_fuzzer = HeuristicGrammarFuzzer(
    TOML_GRAMMAR,
    max_nonterminals=10
)

heuristic_inputs = [
    heuristic_fuzzer.fuzz() for _ in range(N)
]

print("Total heuristic inputs:", len(heuristic_inputs))
print("\nFirst 5 generated inputs:")

for i, test_input in enumerate(heuristic_inputs[:5], start=1):
    print(f"{i}: {test_input}")


Total heuristic inputs: 1000

First 5 generated inputs:
1: title = "Alice"
settings = { name = 3.14, details = { name = 42 } }
2: title = "Bob"
3: title = "Hello TOML"
items = []
4: count = 42
5: title = "Alice"
items = ["Alice",]


## 10. Track grammar coverage for the heuristic

I combine the grammar-coverage tracker with the heuristic fuzzer. I also compare acceptance rates using the same number of normal and heuristic inputs.

In [10]:

from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer

class TrackedHeuristicFuzzer(TrackingGrammarCoverageFuzzer, HeuristicGrammarFuzzer):
    pass

random.seed(42)

tracked_heuristic = TrackedHeuristicFuzzer(
    TOML_GRAMMAR,
    max_nonterminals=10
)

heuristic_inputs = [
    tracked_heuristic.fuzz() for _ in range(1000)
]

used = tracked_heuristic.expansion_coverage()
total = tracked_heuristic.max_expansion_coverage()

heuristic_grammar_coverage = len(used) / len(total) * 100

print("HEURISTIC GRAMMAR COVERAGE")
print("Generated inputs:", len(heuristic_inputs))
print("Total productions:", len(total))
print("Used productions:", len(used))
print(f"Grammar coverage: {heuristic_grammar_coverage:.2f}%")
print("Missing productions:", tracked_heuristic.missing_expansion_coverage())


def acceptance_counts(inputs):
    accepted_count = 0
    rejected_count = 0
    unexpected_count = 0

    for text in inputs:
        try:
            tomllib.loads(text)
            accepted_count += 1
        except tomllib.TOMLDecodeError:
            rejected_count += 1
        except Exception:
            unexpected_count += 1

    return accepted_count, rejected_count, unexpected_count


normal_acceptance = acceptance_counts(normal_inputs)
heuristic_acceptance = acceptance_counts(heuristic_inputs)

print("\nACCEPTANCE COMPARISON")
print(f"{'Metric':<22} {'Normal':>12} {'Heuristic':>12}")
print("-" * 48)
print(f"{'Accepted':<22} {normal_acceptance[0]:>12} {heuristic_acceptance[0]:>12}")
print(f"{'Rejected':<22} {normal_acceptance[1]:>12} {heuristic_acceptance[1]:>12}")
print(f"{'Unexpected errors':<22} {normal_acceptance[2]:>12} {heuristic_acceptance[2]:>12}")
print(f"{'Acceptance rate':<22} {normal_acceptance[0] / len(normal_inputs) * 100:>11.2f}% {heuristic_acceptance[0] / len(heuristic_inputs) * 100:>11.2f}%")


HEURISTIC GRAMMAR COVERAGE
Generated inputs: 1000
Total productions: 44
Used productions: 44
Grammar coverage: 100.00%
Missing productions: set()

ACCEPTANCE COMPARISON
Metric                       Normal    Heuristic
------------------------------------------------
Accepted                       1000         1000
Rejected                          0            0
Unexpected errors                 0            0
Acceptance rate             100.00%      100.00%


## 11. Measure heuristic parser code coverage

I run the heuristic inputs through the same TOML parser and coverage settings as the normal fuzzer.

In [11]:

import coverage
import tomllib

heuristic_cov = coverage.Coverage(
    source=["tomllib"],
    branch=False,
    data_file=None,
    cover_pylib=True
)

heuristic_cov.start()

try:
    for test_input in heuristic_inputs:
        try:
            tomllib.loads(test_input)
        except tomllib.TOMLDecodeError:
            pass
finally:
    heuristic_cov.stop()

print("HEURISTIC FUZZER - CODE COVERAGE")
heuristic_code_coverage = heuristic_cov.report()

print(f"\nTotal line coverage: {heuristic_code_coverage:.2f}%")


HEURISTIC FUZZER - CODE COVERAGE
Name                                                                            Stmts   Miss  Cover
---------------------------------------------------------------------------------------------------
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\__init__.py       3      3     0%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_parser.py      464    254    45%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_re.py           35     32     9%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_types.py         4      4     0%
---------------------------------------------------------------------------------------------------
TOTAL                                                                             506    293    42%

Total line coverage: 42.09%


## 12. Compare how complex the inputs are

I measure average nesting depth, maximum depth, and average input length for normal and heuristic inputs.

In [12]:

from statistics import mean
import tomllib

def nesting_depth(data):
    if isinstance(data, dict):
        return 1 + max((nesting_depth(v) for v in data.values()), default=0)

    if isinstance(data, list):
        return 1 + max((nesting_depth(v) for v in data), default=0)

    return 0


def measure_complexity(inputs):
    depths = []
    lengths = []

    for text in inputs:
        try:
            data = tomllib.loads(text)
            depths.append(nesting_depth(data))
            lengths.append(len(text))
        except tomllib.TOMLDecodeError:
            pass

    if not depths:
        return 0, 0, 0

    return mean(depths), max(depths), mean(lengths)


normal = measure_complexity(normal_inputs)
heuristic = measure_complexity(heuristic_inputs)

print(f"{'Metric':<22} {'Normal':>12} {'Heuristic':>12}")
print("-" * 48)

for i, metric in enumerate(["Average depth", "Maximum depth", "Average length"]):
    print(f"{metric:<22} {normal[i]:>12.2f} {heuristic[i]:>12.2f}")


Metric                       Normal    Heuristic
------------------------------------------------
Average depth                  2.14         2.90
Maximum depth                 10.00        18.00
Average length                42.42        60.85


## 13. Generate mutated test inputs

I select up to 20 longer inputs from the normal grammar run and let `MutationFuzzer` change their characters. The mutated inputs may contain invalid TOML.

In [13]:

import random
from fuzzingbook.MutationFuzzer import MutationFuzzer

seed_inputs = [s for s in normal_inputs if len(s) > 10][:20]

random.seed(42)

mutation_fuzzer = MutationFuzzer(
    seed=seed_inputs,
    min_mutations=1,
    max_mutations=3
)

for _ in seed_inputs:
    mutation_fuzzer.fuzz()

mutation_inputs = [
    mutation_fuzzer.fuzz() for _ in range(1000)
]

print("Mutation inputs:", len(mutation_inputs))
print("\nFirst 5 mutated inputs:")

for i, text in enumerate(mutation_inputs[:5], 1):
    print(f"{i}: {repr(text)}")


Mutation inputs: 1000

First 5 mutated inputs:
1: '[owner]\nname = "Hgllo TOML"\nage = -3'
2: 'titLe = "Alice"\nenabled = false'
3: 'title = "Bob"\x0bcount = 42'
4: '[database]\nserver = "Alice"\nprt = 1\n[database.connection]\nmax_connections = 0\nenabled = true'
5: 'title = "HelloY TOML"\nenabled = fal'


## 14. Measure mutation coverage and acceptance

I test the mutated strings with TOML and measure parser code coverage. I count accepted inputs, expected syntax rejections, and unexpected exceptions.

In [14]:

import coverage
import tomllib

mutation_cov = coverage.Coverage(
    source=["tomllib"],
    branch=False,
    data_file=None,
    cover_pylib=True
)

accepted = 0
rejected = 0
unexpected = 0

mutation_cov.start()

try:
    for test_input in mutation_inputs:
        try:
            tomllib.loads(test_input)
            accepted += 1

        except tomllib.TOMLDecodeError:
            rejected += 1

        except Exception as error:
            unexpected += 1
            print(type(error).__name__, error)

finally:
    mutation_cov.stop()

print("MUTATION FUZZER - CODE COVERAGE")
mutation_code_coverage = mutation_cov.report()

print(f"\nTotal line coverage: {mutation_code_coverage:.2f}%")
print("Accepted:", accepted)
print("Rejected:", rejected)
print("Unexpected errors:", unexpected)
print(f"Acceptance rate: {accepted / len(mutation_inputs) * 100:.2f}%")


MUTATION FUZZER - CODE COVERAGE
Name                                                                            Stmts   Miss  Cover
---------------------------------------------------------------------------------------------------
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\__init__.py       3      3     0%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_parser.py      464    208    55%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_re.py           35     32     9%
C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\tomllib\_types.py         4      4     0%
---------------------------------------------------------------------------------------------------
TOTAL                                                                             506    247    51%

Total line coverage: 51.19%
Accepted: 356
Rejected: 644
Unexpected errors: 0
Acceptance rate: 35.60%


## 15. Time how long parsing takes

I time the TOML parser on each generated input set. Each run is repeated three times, and I report the median. This measures parsing time only.

In [15]:

import time
from statistics import median
import tomllib

def measure_time(inputs, repeats=3):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()

        for text in inputs:
            try:
                tomllib.loads(text)
            except tomllib.TOMLDecodeError:
                pass

        times.append(time.perf_counter() - start)

    return median(times)


normal_time = measure_time(normal_inputs)
heuristic_time = measure_time(heuristic_inputs)
mutation_time = measure_time(mutation_inputs)

print("PARSER EXECUTION TIME")
print(f"Normal:    {normal_time:.4f} seconds")
print(f"Heuristic: {heuristic_time:.4f} seconds")
print(f"Mutation:  {mutation_time:.4f} seconds")


PARSER EXECUTION TIME
Normal:    0.0325 seconds
Heuristic: 0.0338 seconds
Mutation:  0.0223 seconds


## 16. Time how long generation takes

I create fresh normal, heuristic, and mutation fuzzers and measure how long each takes to produce 1,000 inputs. Mutation timing does not include the creation of its original grammar-based seeds.

In [16]:

import time
import random
from fuzzingbook.GrammarFuzzer import GrammarFuzzer
from fuzzingbook.MutationFuzzer import MutationFuzzer

N = 1000

def generation_time(fuzzer, n):
    start = time.perf_counter()

    for _ in range(n):
        fuzzer.fuzz()

    return time.perf_counter() - start


random.seed(42)
normal_gen = GrammarFuzzer(TOML_GRAMMAR, max_nonterminals=10)
normal_gen_time = generation_time(normal_gen, N)

random.seed(42)
heuristic_gen = HeuristicGrammarFuzzer(TOML_GRAMMAR, max_nonterminals=10)
heuristic_gen_time = generation_time(heuristic_gen, N)

seed_inputs = [s for s in normal_inputs if len(s) > 10][:20]
mutation_gen = MutationFuzzer(seed=seed_inputs, min_mutations=1, max_mutations=3)

for _ in seed_inputs:
    mutation_gen.fuzz()

random.seed(42)
mutation_gen_time = generation_time(mutation_gen, N)

print("INPUT GENERATION TIME")
print(f"Normal:    {normal_gen_time:.4f} seconds")
print(f"Heuristic: {heuristic_gen_time:.4f} seconds")
print(f"Mutation:  {mutation_gen_time:.4f} seconds")


INPUT GENERATION TIME
Normal:    0.7904 seconds
Heuristic: 4.9265 seconds
Mutation:  0.0083 seconds


## 17. Summarize time, coverage, and acceptance

In [18]:

normal_total = normal_gen_time + normal_time
heuristic_total = heuristic_gen_time + heuristic_time
mutation_total = mutation_gen_time + mutation_time

print("TOTAL TESTING TIME (1000 INPUTS)")
print("-" * 40)
print(f"Normal:    {normal_total:.4f} seconds")
print(f"Heuristic: {heuristic_total:.4f} seconds")
print(f"Mutation:  {mutation_total:.4f} seconds")

print("\nGRAMMAR COVERAGE")
print(f"Normal:    {normal_grammar_coverage:.2f}%")
print(f"Heuristic: {heuristic_grammar_coverage:.2f}%")

print("\nPARSER CODE COVERAGE")
print(f"Normal:    {normal_code_coverage:.2f}%")
print(f"Heuristic: {heuristic_code_coverage:.2f}%")
print(f"Mutation:  {mutation_code_coverage:.2f}%")

print("\nACCEPTANCE RATE")
print(f"Normal:    {normal_acceptance[0] / len(normal_inputs) * 100:.2f}%")
print(f"Heuristic: {heuristic_acceptance[0] / len(heuristic_inputs) * 100:.2f}%")
print(f"Mutation:  {accepted / len(mutation_inputs) * 100:.2f}%")


TOTAL TESTING TIME (1000 INPUTS)
----------------------------------------
Normal:    0.8229 seconds
Heuristic: 4.9602 seconds
Mutation:  0.0306 seconds

GRAMMAR COVERAGE
Normal:    100.00%
Heuristic: 100.00%

PARSER CODE COVERAGE
Normal:    42.09%
Heuristic: 42.09%
Mutation:  51.19%

ACCEPTANCE RATE
Normal:    100.00%
Heuristic: 100.00%
Mutation:  35.60%
